# observational uncertainty

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

# Observational uncertainty: audit and a validated integration identity

The six simulation-comparison models remain noise-free. This notebook separates their conditional posterior uncertainty from uncertainty in real Local Group measurements. It is an executed methodological audit, **not a new Local Group mass result**. See [the decision record](../../observational_uncertainty.ipynb).


In [ ]:
import os, sys, json, hashlib, pickle
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
candidates = [Path(os.environ.get('LG_PROJECT', Path.cwd())), Path.cwd(), *Path.cwd().parents]
PROJECT = next(p.resolve() for p in candidates if (p/'src/lg_npe').is_dir())
sys.path.insert(0, str(PROJECT/'src'))
from lg_npe.final_design import *
from lg_npe.registry import environment_record, write_json
from lg_npe.features import feature_names
from lg_npe.evaluation import posterior_metrics
from lg_npe.inference import posterior_draws
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Project:', PROJECT)

## Observational inputs recovered

Distance and velocity errors are not all independent Gaussian distributions. In particular, a tangential-speed upper limit is not a standard deviation, a positive speed is the norm of uncertain components, and stellar-population-model ranges are not confidence intervals. The historical 765±15 kpc value includes a chosen widening; Li et al.'s actual Cepheid result is 761±11 kpc heliocentric.


In [ ]:
observations=pd.DataFrame([
    ['distance','761 +/- 11 kpc heliocentric','24.407 +/- .032 distance modulus','Li et al. 2021','https://arxiv.org/abs/2107.08029'],
    ['vr','-109.3 +/- 4.4 km/s','physical relative; solar correction','van der Marel et al. 2012','https://arxiv.org/abs/1205.6864'],
    ['vt HST','17 km/s; <34.3 at 1 sigma','not a Gaussian sigma; point-only legacy scenario','van der Marel et al. 2012','https://arxiv.org/abs/1205.6864'],
    ['vt Gaia','82.4 +/- 31.2 km/s','speed distribution and joint covariance required','Salomon et al. 2021','https://arxiv.org/abs/2012.09204'],
    ['MW stellar mass','(6.08 +/- 1.14) x 1e10 Msun','Kroupa IMF; shared structure assumptions','Licquia & Newman 2015','https://arxiv.org/abs/1407.1078'],
    ['M31 stellar mass','(10--15) x 1e10 Msun','population-model range; not Gaussian uncertainty','Tamm et al. 2012','https://arxiv.org/abs/1208.5712'],
],columns=['observable','value','distribution_caveat','source','url'])
display(observations)
OUTPUT=PROJECT/'out/final_observation_audit'/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
observations.to_csv(OUTPUT/'observational_sources.csv',index=False)

## What distribution does averaging posteriors estimate?

For latent catalogue features x, observations y and measurement kernel K(y|x), conditional independence gives

$$p_S(\theta|y)=\frac{\int p_S(\theta|x)K(y|x)p_S(x)\,dx}{\int K(y|x)p_S(x)\,dx}.$$

Averaging a noiseless NPE over an arbitrary perturbation density r(x|y) instead estimates a different mixture. It is correct only if r equals the latent-feature posterior under the same selected population, or is weighted by K(y|x)p_S(x)/r(x|y). An external observation posterior contains its own prior; do not silently count that prior twice.

Noise-aware training simulates the measurement process forward. It is a suitable separate application once the joint likelihood is defined. Supplying uncertainty as another input is useful only if the training observation process varies in known ways.


In [ ]:
from scipy.stats import norm
from lg_npe.observation import normalized_importance_weights, gaussian_linear_observation
# Synthetic demonstration only: theta|x ~ N(x, .2^2), x~N(0,1), y|x~N(x,1), y=2.
# The correct x|y is N(1,.5), while naive measurement perturbation is N(2,1).
rng=np.random.default_rng(261001); x=rng.normal(2.,1.,200000)
w,ess=normalized_importance_weights(norm.logpdf(2.,loc=x),norm.logpdf(x),norm.logpdf(x,loc=2.))
mu,var=gaussian_linear_observation(1.,1.,2.)
weighted_mean=np.dot(w,x); weighted_variance=np.dot(w,(x-weighted_mean)**2)+.2**2
assert abs(weighted_mean-mu)<.02
assert abs(weighted_variance-(var+.2**2))<.02
check=pd.DataFrame([
    ['analytic',mu,var+.2**2],['weighted integration',weighted_mean,weighted_variance],
    ['naive perturbation',x.mean(),x.var()+.2**2]],columns=['method','theta_mean','theta_variance'])
display(check); print('Importance effective sample size:',round(ess))
check.to_csv(OUTPUT/'synthetic_integration_validation.csv',index=False)
grid=np.linspace(-2,5,500)
fig,ax=plt.subplots(figsize=(7,4),layout='constrained')
ax.plot(grid,norm.pdf(grid,mu,np.sqrt(var+.2**2)),label='correct marginalized posterior')
ax.plot(grid,norm.pdf(grid,2,np.sqrt(1+.2**2)),ls='--',label='naive perturbation mixture')
ax.set(xlabel='synthetic theta (not halo mass)',ylabel='density'); ax.legend()
fig.savefig(OUTPUT/'integration_validation.png',dpi=200)
fig.savefig(OUTPUT/'integration_validation.svg'); plt.show()

## Velocity conversion and covariance

The model features are peculiar velocities. Use vr,pec = vr,physical − H0 r/1000 for every joint observation draw. Even if physical r and vr were independent, this creates Cov(r,vr,pec)=−H0 Var(r)/1000. The legacy helper preserves this induced correlation. Additional distance/proper-motion/solar-motion covariance must come from measurements, not a fabricated matrix. For combined models, any future application must define a physical observation kernel for each simulation's H0 before mixing; averaging H0 without a model is not justified.


In [ ]:
sys.path.insert(0,str(PROJECT/'analogues'))
from review_fixes import lg_to_peculiar
# Synthetic covariance check only, not a fitted Local Group observation distribution.
rng=np.random.default_rng(42)
physical=np.column_stack([rng.normal(765,15,200000),rng.normal(-109.3,4.4,200000),np.full(200000,82.4)])
peculiar=lg_to_peculiar(physical,['separation','vr','vt'],68.1)
expected=np.cov(physical[:,0],physical[:,1],ddof=0)[0,1]-68.1*np.var(physical[:,0])/1000
actual=np.cov(peculiar[:,0],peculiar[:,1],ddof=0)[0,1]
assert abs(actual-expected)<1e-3
print('Transformation covariance check:',actual,expected)
write_json(OUTPUT/'manifest.json',dict(status='methodology_validated_LG_inference_deferred',
    synthetic_seed=261001,synthetic_ess=ess,environment=environment_record(),
    observation_model='not yet specified; no real posterior inferred',
    decision_sha256=sha256(PROJECT/'OBSERVATIONAL_UNCERTAINTY_DECISION_2026-10-01.md')))

## Decision and remaining application requirements

Do not call the historical perturb-and-average outputs exact uncertainty-marginalized Local Group posteriors. They remain sensitivity calculations under adopted input distributions. The synthetic example validates the correction algebra, not the real observation model.

Before a final application, recover a reproducible joint observational likelihood or samples with their reference prior, resolve physical/peculiar velocity conventions, check selected-population support, and reconcile stellar mass definitions/IMFs if stellar conditioning is used. The scalar stellar-mass summaries above are insufficient for a validated joint likelihood. Therefore no final Local Group posterior is forced here. Keep observational error, domain sensitivity, calibration bootstrap uncertainty and finder systematics separate.
